# 🌕 ENIGMA 2.0 — Lunar Mapper Batch Pipeline
### Automated Boulder & Landslide Detection — Chandrayaan-2 OHRC + TMC-2 DTM
**Team:** WattWatchers

This notebook processes **all images in a folder** and saves each output (CSVs, annotated maps) into `outputs/<image_name>/`


## 📦 Step 0 — Install Dependencies

In [15]:
import sys
!{sys.executable} -m pip install numpy opencv-python scikit-image scipy matplotlib pandas pyyaml tqdm --quiet
print("✅ All packages installed")


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✅ All packages installed


## 📥 Step 1 — Imports

In [16]:
import numpy as np
import cv2, os, time, warnings, glob
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from scipy.ndimage import gaussian_filter, generic_filter
from skimage import morphology, measure, filters
from skimage.filters.rank import entropy as rank_entropy
from skimage.morphology import disk, erosion, dilation
from skimage.feature import blob_log
from skimage.filters import gaussian
warnings.filterwarnings('ignore')
print("✅ All imports successful")

✅ All imports successful


## ⚙️ Step 2 — Configuration

In [17]:
CFG = {
    'data': {
        'pixel_resolution_m': 0.3,   # OHRC: 0.3 m/pixel
        'tile_size':          1000,  # Process in NxN tiles
        'tile_overlap':       50,    # Overlap to avoid edge artifacts
    },
    'boulder_detection': {
        'method':           'hybrid',  # 'hough', 'blob', or 'hybrid'
        'hough_dp':          1,
        'hough_min_dist':    15,
        'hough_param1':      50,
        'hough_param2':      25,       # Lower = more detections
        'hough_min_radius':  3,
        'hough_max_radius':  80,
        'blob_min_sigma':    2,
        'blob_max_sigma':    30,
        'blob_num_sigma':    10,
        'blob_threshold':    0.05,
        'min_diameter_m':    1.0,
        'max_diameter_m':    50.0,
        'contrast_threshold': 20,
    },
    'landslide_detection': {
        'slope_threshold_deg':      12.0,
        'texture_percentile':       75,
        'texture_disk_radius':      5,
        'min_area_pixels':          300,
        'max_area_pixels':          5_000_000,
        'elongation_ratio':         0.65,
        'morphology_close_radius':  3,
        'morphology_open_radius':   2,
    },
    'age_classification': {
        'fresh_gradient_multiplier':  1.8,
        'fresh_roughness_threshold':  4.0,
        'fresh_score_threshold':      0.55,
    },
    'visualization': {
        'boulder_color':  [255,  50,  50],
        'landslide_color':[50,  150, 255],
        'source_color':   [0,   255, 100],
        'recent_color':   [255, 200,   0],
        'ancient_color':  [180, 180, 180],
        'dpi':     150,
        'figsize': [20, 20],
    }
}
print("✅ Config loaded")

✅ Config loaded


## 📁 Step 3 — Batch Settings
> Set `INPUT_FOLDER` to the folder containing your images.  
> Each image gets its own subfolder inside `OUTPUT_FOLDER`.


In [18]:
# ── SET THESE ───────────────────────────────────────────────────────────────
INPUT_FOLDER  = 'data/raw'          # Folder with your .png/.img/.tif images
OUTPUT_FOLDER = 'outputs'           # Root output folder
DTM_FOLDER    = None                # Folder with matching DTM files (optional)
#                                     DTM must have same name as image, e.g.:
#                                     image: ohrc_01.png → dtm: ohrc_01_dtm.tif
#                                     Set to None if you have no DTM files
MAX_SIZE      = 1024                # Resize longest side to this (faster)
#                                     Set to None for full resolution
DEMO          = False               # Set True to run on synthetic data instead
# ────────────────────────────────────────────────────────────────────────────

SUPPORTED_EXTS = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.img', '.npy')

if DEMO:
    image_paths = ['__DEMO__']
    print('Mode: DEMO — will run on 1 synthetic image')
else:
    if not os.path.exists(INPUT_FOLDER):
        os.makedirs(INPUT_FOLDER, exist_ok=True)
        print(f'⚠️  Created empty folder: {INPUT_FOLDER}')
        print(f'    Put your .png / .img / .tif images inside it, then re-run this cell.')
        image_paths = []
    else:
        image_paths = sorted([
            f for f in glob.glob(os.path.join(INPUT_FOLDER, '*'))
            if os.path.splitext(f)[1].lower() in SUPPORTED_EXTS
        ])
        print(f'📂 Input folder : {INPUT_FOLDER}')
        print(f'📂 Output folder: {OUTPUT_FOLDER}')
        print(f'🖼️  Found {len(image_paths)} image(s):')
        for p in image_paths:
            sz = os.path.getsize(p) / (1024*1024)
            print(f'   • {os.path.basename(p)}  ({sz:.1f} MB)')
        if not image_paths:
            print(f'   No supported images found in {INPUT_FOLDER}')
            print(f'   Supported formats: {SUPPORTED_EXTS}')


📂 Input folder : data/raw
📂 Output folder: outputs
🖼️  Found 11 image(s):
   • ohrc_image_01.png  (9.8 MB)
   • ohrc_image_02.png  (10.4 MB)
   • ohrc_image_03.png  (8.4 MB)
   • ohrc_image_04.png  (7.8 MB)
   • ohrc_image_05.png  (9.4 MB)
   • ohrc_image_06.png  (0.3 MB)
   • ohrc_image_07..png  (0.3 MB)
   • ohrc_image_08..png  (0.3 MB)
   • ohrc_image_09.png  (0.3 MB)
   • ohrc_image_10.png  (0.3 MB)
   • ohrc_image_11.png  (0.5 MB)


## 🔧 Step 4 — Define All Detection Functions

In [19]:
# ── Image loading & preprocessing ────────────────────────────────────────────
def normalize_image(img):
    img = img.copy().astype(np.float32)
    img[np.isnan(img)] = np.nanmedian(img)
    mn, mx = np.nanpercentile(img, 2), np.nanpercentile(img, 98)
    img = np.clip(img, mn, mx)
    return ((img - mn) / (mx - mn) * 255).astype(np.uint8)

def smart_resize(img, dem, max_size):
    if max_size is None: return img, dem
    h, w = img.shape
    if max(h, w) <= max_size: return img, dem
    scale = max_size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    print(f'    Resizing {w}x{h} → {new_w}x{new_h} (scale={scale:.2f})')
    img_r = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    dem_r = cv2.resize(dem.astype(np.float32), (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    return img_r, dem_r

def load_img_file(path):
    ext = path.lower().split('.')[-1]
    if ext in ('png','jpg','jpeg'):
        arr = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if arr is None: raise FileNotFoundError(f'Cannot read: {path}')
        return arr.astype(np.float32)
    elif ext in ('tif','tiff'):
        try:
            import rasterio
            with rasterio.open(path) as src:
                return src.read(1).astype(np.float32)
        except ImportError:
            arr = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            return arr.astype(np.float32)
    elif ext == 'img':
        raw  = np.fromfile(path, dtype=np.uint8)
        size = int(np.sqrt(len(raw)))
        print(f'    .img: {len(raw)} bytes → {size}x{size}px')
        return raw[:size*size].reshape(size, size).astype(np.float32)
    elif ext == 'npy':
        return np.load(path).astype(np.float32)
    else:
        raise ValueError(f'Unsupported: .{ext}')

def generate_synthetic_data(size=1024):
    print('    Generating synthetic lunar terrain...')
    rng = np.random.default_rng(42)
    dem = gaussian_filter(rng.random((size, size)) * 200, sigma=80)
    for _ in range(15):
        cx, cy = rng.integers(100, size-100, 2)
        r = rng.integers(20, 120)
        yy, xx = np.ogrid[:size, :size]
        dist = np.sqrt((xx-cx)**2 + (yy-cy)**2)
        dem[dist < r] -= (r - dist[dist < r]) * 0.8
    dem = dem.astype(np.float32)
    base = normalize_image(dem).astype(np.float32)
    base += rng.normal(0, 8, base.shape)
    for _ in range(80):
        bx, by = rng.integers(20, size-20, 2)
        br = rng.integers(3, 18)
        yy, xx = np.ogrid[:size, :size]
        dist = np.sqrt((xx-bx)**2 + (yy-by)**2)
        base[dist < br] += 60
        base[((xx-bx) > 0) & ((xx-bx) < br+5) & (np.abs(yy-by) < br)] -= 30
    for _ in range(8):
        sx, sy = rng.integers(100, size-200, 2)
        length = rng.integers(100, 400)
        width  = rng.integers(10, 40)
        angle  = rng.uniform(30, 150)
        for t in range(length):
            tx = int(sx + t * np.cos(np.radians(angle)))
            ty = int(sy + t * np.sin(np.radians(angle)))
            if 0 <= tx < size and 0 <= ty < size:
                base[max(0,ty-width//2):min(size,ty+width//2),
                     max(0,tx-3):min(size,tx+3)] += 40
        dem[sy:sy+width, sx:sx+length//2] += 15
    return np.clip(base, 0, 255).astype(np.uint8), dem

# ── Tiling ────────────────────────────────────────────────────────────────────
def tile_image(img, dem, tile_size=1000, overlap=50):
    h, w = img.shape
    tiles = []
    r = 0
    while r < h:
        c = 0
        while c < w:
            r_end = min(r + tile_size, h)
            c_end = min(c + tile_size, w)
            tiles.append((img[r:r_end, c:c_end], dem[r:r_end, c:c_end], r, c))
            c += tile_size - overlap
        r += tile_size - overlap
    return tiles

# ── Boulder detection ─────────────────────────────────────────────────────────
def _local_contrast(img, cx, cy, r):
    h, w = img.shape
    yy, xx = np.ogrid[:h, :w]
    dist = np.sqrt((xx-cx)**2 + (yy-cy)**2)
    inner = img[dist < r]; outer = img[(dist >= r) & (dist < r*2.5)]
    if len(inner)==0 or len(outer)==0: return 0.0
    return abs(float(np.mean(inner)) - float(np.mean(outer)))

def detect_boulders_hough(img_gray, cfg):
    blurred = cv2.GaussianBlur(img_gray, (5,5), 1.5)
    circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT,
        dp=cfg['hough_dp'], minDist=cfg['hough_min_dist'],
        param1=cfg['hough_param1'], param2=cfg['hough_param2'],
        minRadius=cfg['hough_min_radius'], maxRadius=cfg['hough_max_radius'])
    if circles is None: return []
    result = []
    for (x,y,r) in np.round(circles[0]).astype(int):
        if _local_contrast(img_gray.astype(float),x,y,r) >= cfg['contrast_threshold']:
            result.append((int(x),int(y),int(r)))
    return result

def detect_boulders_blob(img_gray, cfg):
    img_norm = img_gray.astype(float)/255.0
    img_norm = gaussian(img_norm, sigma=1)
    blobs = blob_log(img_norm, min_sigma=cfg['blob_min_sigma'],
        max_sigma=cfg['blob_max_sigma'], num_sigma=cfg['blob_num_sigma'],
        threshold=cfg['blob_threshold'])
    result = []
    for blob in blobs:
        y, x, sigma = blob
        r = int(np.round(sigma * np.sqrt(2)))
        x, y = int(x), int(y)
        if _local_contrast(img_gray.astype(float),x,y,r) >= cfg['contrast_threshold']:
            result.append((x,y,r))
    return result

def _deduplicate(detections, min_dist=15):
    if not detections: return []
    detections = sorted(detections, key=lambda d: -d[2])
    kept = []
    for d in detections:
        cx, cy, cr = d
        if not any(np.sqrt((cx-k[0])**2+(cy-k[1])**2) < max(cr,k[2]) for k in kept):
            kept.append(d)
    return kept

def detect_boulders(img, cfg, pixel_res=0.3):
    method = cfg.get('method','hybrid')
    detections = []
    if method in ('hough','hybrid'): detections.extend(detect_boulders_hough(img, cfg))
    if method in ('blob','hybrid'):  detections.extend(detect_boulders_blob(img, cfg))
    detections = _deduplicate(detections)
    min_r = cfg['min_diameter_m'] / (2*pixel_res)
    max_r = cfg['max_diameter_m'] / (2*pixel_res)
    detections = [(x,y,r) for x,y,r in detections if min_r <= r <= max_r]
    return [{'x_pixel':x,'y_pixel':y,'radius_px':r,
             'diameter_m':round(r*2*pixel_res,2),
             'contrast':round(_local_contrast(img.astype(float),x,y,r),2)}
            for (x,y,r) in detections]

# ── Landslide detection ───────────────────────────────────────────────────────
def compute_slope(dem, pixel_res=5.0):
    if np.nanstd(dem) < 0.1: return np.zeros_like(dem, dtype=np.float32)
    gy, gx = np.gradient(dem, pixel_res)
    slope  = np.arctan(np.sqrt(gx**2+gy**2)) * (180.0/np.pi)
    return np.nan_to_num(slope, nan=0.0).astype(np.float32)

def compute_texture_entropy(img, disk_radius=5):
    img_u8 = img.astype(np.uint8) if img.dtype!=np.uint8 else img
    return rank_entropy(img_u8, disk(disk_radius)).astype(np.float32)

def detect_landslides(img, dem, cfg, pixel_res=0.3):
    slope   = compute_slope(dem, pixel_res)
    texture = compute_texture_entropy(img, cfg['texture_disk_radius'])
    slope_mask   = slope > cfg['slope_threshold_deg']
    tex_thresh   = np.percentile(texture, cfg['texture_percentile'])
    texture_mask = texture > tex_thresh
    candidate    = slope_mask & texture_mask
    cr, or_ = cfg['morphology_close_radius'], cfg['morphology_open_radius']
    if cr > 0: candidate = morphology.binary_closing(candidate, disk(cr))
    if or_>0: candidate = morphology.binary_opening(candidate, disk(or_))
    candidate = morphology.remove_small_objects(candidate, min_size=cfg['min_area_pixels'])
    candidate = morphology.remove_small_holes(candidate, area_threshold=cfg['min_area_pixels']//2)
    labels  = measure.label(candidate.astype(bool))
    regions = measure.regionprops(labels, intensity_image=dem)
    landslides = []
    for region in regions:
        area_px = region.area
        if area_px < cfg['min_area_pixels'] or area_px > cfg['max_area_pixels']: continue
        if region.minor_axis_length == 0: continue
        elongation = region.minor_axis_length / region.major_axis_length
        if elongation > cfg['elongation_ratio']: continue
        coords    = region.coords
        elevs     = dem[coords[:,0], coords[:,1]]
        max_idx   = np.argmax(elevs)
        sr, sc    = coords[max_idx]
        cr2, cc2  = region.centroid
        landslides.append({
            'source_row':float(sr),'source_col':float(sc),
            'centroid_row':float(cr2),'centroid_col':float(cc2),
            'area_px':int(area_px),'area_m2':round(area_px*(pixel_res**2),1),
            'elongation':round(elongation,3),
            'major_axis_m':round(region.major_axis_length*pixel_res,1),
            'minor_axis_m':round(region.minor_axis_length*pixel_res,1),
            'orientation_deg':round(float(region.orientation)*180/np.pi,1),
            'coords':coords
        })
    return landslides

# ── Age classification ────────────────────────────────────────────────────────
def compute_freshness_score(img, dem, ls, cfg, pixel_res=0.3):
    coords = ls['coords']
    h, w   = img.shape
    mask   = np.zeros((h,w), dtype=bool)
    rows   = np.clip(coords[:,0].astype(int), 0, h-1)
    cols   = np.clip(coords[:,1].astype(int), 0, w-1)
    mask[rows, cols] = True
    interior = erosion(mask, disk(3))
    boundary = mask & ~interior
    if not np.any(boundary): boundary = mask
    grad = filters.sobel(img.astype(float))
    dilated = dilation(mask, disk(10))
    surround = dilated & ~mask
    mean_b = float(np.mean(grad[boundary])) if np.any(boundary) else 0.0
    mean_s = float(np.mean(grad[surround])) if np.any(surround) else 1.0
    grad_ratio  = mean_b / (mean_s + 1e-6)
    slope       = compute_slope(dem, pixel_res)
    roughness   = float(np.std(slope[rows, cols]))
    inner_mean  = float(np.mean(img[mask].astype(float)))
    outer_mean  = float(np.mean(img[surround].astype(float))) if np.any(surround) else inner_mean
    albedo_c    = abs(inner_mean - outer_mean)
    mult        = cfg.get('fresh_gradient_multiplier', 1.8)
    rt          = cfg.get('fresh_roughness_threshold', 4.0)
    grad_score  = min(1.0, grad_ratio / (mult*2))
    rough_score = min(1.0, roughness  / (rt*2))
    alb_score   = min(1.0, albedo_c   / 50.0)
    freshness   = 0.5*grad_score + 0.3*rough_score + 0.2*alb_score
    threshold   = cfg.get('fresh_score_threshold', 0.55)
    result = ls.copy()
    result.update({'gradient_ratio':round(grad_ratio,4),'slope_roughness_deg':round(roughness,3),
                   'albedo_contrast':round(albedo_c,2),'freshness_score':round(freshness,4),
                   'age_class':'recent' if freshness >= threshold else 'ancient'})
    return result

def tag_boulders_near_landslides(boulders, landslides, proximity_px=50):
    ls_pixels = {}
    for ls in landslides:
        age = ls.get('age_class','unknown')
        for (r,c) in ls.get('coords',[]):
            ls_pixels[(int(r),int(c))] = age
    updated = []
    for b in boulders:
        bx, by = b['x_pixel'], b['y_pixel']
        nearby, nearby_age = False, 'none'
        for dr in range(-proximity_px, proximity_px, 5):
            for dc in range(-proximity_px, proximity_px, 5):
                if (by+dr, bx+dc) in ls_pixels:
                    nearby, nearby_age = True, ls_pixels[(by+dr, bx+dc)]
                    break
            if nearby: break
        b2 = b.copy(); b2['near_landslide']=nearby; b2['landslide_age']=nearby_age
        updated.append(b2)
    return updated

# ── Visualization helpers ─────────────────────────────────────────────────────
def _to_rgb(img):
    return np.stack([img,img,img], axis=-1) if img.ndim==2 else img.copy()

def _blend(canvas, overlay, alpha):
    mask = np.any(overlay > 0, axis=-1)
    blended = canvas.copy()
    blended[mask] = ((1-alpha)*canvas[mask].astype(float)+alpha*overlay[mask].astype(float)).astype(np.uint8)
    return blended

def draw_landslide_fills(img_rgb, landslides, cfg, fill_alpha=0.45, boundary_alpha=0.9):
    canvas = img_rgb.copy()
    h, w   = canvas.shape[:2]
    rc = np.array(cfg['recent_color'],  dtype=np.uint8)
    ac = np.array(cfg['ancient_color'], dtype=np.uint8)
    for ls in landslides:
        coords = ls.get('coords', np.zeros((0,2),dtype=int))
        if len(coords)==0: continue
        color = rc if ls.get('age_class')=='recent' else ac
        rows  = np.clip(coords[:,0].astype(int), 0, h-1)
        cols  = np.clip(coords[:,1].astype(int), 0, w-1)
        mask  = np.zeros((h,w), dtype=bool)
        mask[rows,cols] = True
        ov = np.zeros_like(canvas); ov[mask] = color
        canvas = _blend(canvas, ov, fill_alpha)
        interior = erosion(mask, disk(2))
        boundary = mask & ~interior
        if not np.any(boundary): boundary = mask
        bov = np.zeros_like(canvas); bov[boundary] = color
        canvas = _blend(canvas, bov, boundary_alpha)
        sr, sc = int(ls['source_row']), int(ls['source_col'])
        if 0<=sr<h and 0<=sc<w:
            src_bgr = tuple(int(v) for v in reversed(cfg['source_color']))
            tmp = cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR)
            cv2.drawMarker(tmp,(sc,sr),src_bgr,markerType=cv2.MARKER_CROSS,
                           markerSize=14,thickness=2,line_type=cv2.LINE_AA)
            canvas = cv2.cvtColor(tmp, cv2.COLOR_BGR2RGB)
    return canvas

def draw_boulders_overlay(img_rgb, boulders, cfg, circle_alpha=0.85):
    canvas = img_rgb.copy()
    h, w   = canvas.shape[:2]
    bc = np.array(cfg['boulder_color'], dtype=np.uint8)
    yy_g, xx_g = np.ogrid[:h, :w]
    for b in boulders:
        cx, cy, r = int(b['x_pixel']), int(b['y_pixel']), int(b['radius_px'])
        dist = np.sqrt((xx_g-cx)**2+(yy_g-cy)**2)
        ov = np.zeros_like(canvas); ov[dist<=r] = bc
        canvas = _blend(canvas, ov, 0.25)
        rov = np.zeros_like(canvas); rov[(dist>=r-1)&(dist<=r+1)] = bc
        canvas = _blend(canvas, rov, circle_alpha)
        dot_mask = dist <= max(1.5, r*0.15)
        canvas[dot_mask] = [255,255,100]
    return canvas

print('✅ All functions defined — ready to process images')


✅ All functions defined — ready to process images


## 🚀 Step 5 — Run Batch Processing
> This cell processes **every image** in your input folder one by one.
> Each image gets its own subfolder: `outputs/<image_name>/`


In [20]:
PIXEL_RES = CFG['data']['pixel_resolution_m']
vcfg      = CFG['visualization']
batch_summary = []  # Collect summary across all images

if not image_paths:
    print('⚠️  No images to process. Add images to your INPUT_FOLDER and re-run Step 3.')
else:
    t_batch_start = time.time()
    print(f'{'='*60}')
    print(f'  BATCH START — {len(image_paths)} image(s) to process')
    print(f'{'='*60}\n')

    for img_idx, img_path in enumerate(image_paths):

        # ── Image name & output dir ───────────────────────────────────────
        if img_path == '__DEMO__':
            img_name = 'demo_synthetic'
        else:
            img_name = os.path.splitext(os.path.basename(img_path))[0]

        out_dir     = os.path.join(OUTPUT_FOLDER, img_name)
        csv_dir     = os.path.join(out_dir, 'csv')
        maps_dir    = os.path.join(out_dir, 'maps')
        os.makedirs(csv_dir,  exist_ok=True)
        os.makedirs(maps_dir, exist_ok=True)

        print(f'[{img_idx+1}/{len(image_paths)}] {img_name}')
        print(f'  Output → {out_dir}')
        t_img_start = time.time()

        try:
            # ── Load image ────────────────────────────────────────────────
            _t = time.time()
            if img_path == '__DEMO__':
                img, dem = generate_synthetic_data(MAX_SIZE or 1024)
            else:
                img_raw = load_img_file(img_path)
                img     = normalize_image(img_raw)
                # Look for matching DTM
                if DTM_FOLDER is not None:
                    dtm_candidates = [
                        os.path.join(DTM_FOLDER, img_name + ext)
                        for ext in ('.tif','.tiff','.png','.img','.npy')
                    ] + [
                        os.path.join(DTM_FOLDER, img_name + '_dtm' + ext)
                        for ext in ('.tif','.tiff','.png','.img','.npy')
                    ]
                    dtm_path = next((p for p in dtm_candidates if os.path.exists(p)), None)
                    if dtm_path:
                        dtm_raw = load_img_file(dtm_path)
                        if dtm_raw.shape != img.shape:
                            dtm_raw = cv2.resize(dtm_raw.astype(np.float32),
                                                 (img.shape[1], img.shape[0]),
                                                 interpolation=cv2.INTER_LINEAR)
                        dem = dtm_raw
                        print(f'    DTM loaded: {dtm_path}')
                    else:
                        dem = gaussian_filter(img.astype(np.float32), sigma=15)
                        print(f'    No DTM found — using brightness proxy')
                else:
                    dem = gaussian_filter(img.astype(np.float32), sigma=15)
                # Resize
                img, dem = smart_resize(img, dem, MAX_SIZE)
            print(f'  ⏱ Load: {time.time()-_t:.1f}s  |  Size: {img.shape[1]}x{img.shape[0]}px')

            # ── Tile ──────────────────────────────────────────────────────
            _t = time.time()
            tiles = tile_image(img, dem, CFG['data']['tile_size'], CFG['data']['tile_overlap'])
            print(f'  ⏱ Tiling: {time.time()-_t:.2f}s  |  {len(tiles)} tiles')

            # ── Detect ────────────────────────────────────────────────────
            _t = time.time()
            all_boulders, all_landslides = [], []
            for i_tile, (tile_img, tile_dem, row_off, col_off) in enumerate(tiles):
                _tt = time.time()
                print(f'    Tile {i_tile+1}/{len(tiles)}', end=' ')
                b_list = detect_boulders(tile_img, CFG['boulder_detection'], PIXEL_RES)
                for b in b_list: b['x_pixel']+=col_off; b['y_pixel']+=row_off
                all_boulders.extend(b_list)
                l_list = detect_landslides(tile_img, tile_dem, CFG['landslide_detection'], PIXEL_RES)
                for ls in l_list:
                    ls['source_row']+=row_off;   ls['source_col']+=col_off
                    ls['centroid_row']+=row_off;  ls['centroid_col']+=col_off
                    if ls.get('coords') is not None:
                        ls['coords'] = ls['coords'] + np.array([[row_off,col_off]])
                    ls['_tile_img']=tile_img; ls['_tile_dem']=tile_dem
                    ls['_row_off']=row_off;   ls['_col_off']=col_off
                all_landslides.extend(l_list)
                print(f'| B:{len(b_list)} L:{len(l_list)} ({time.time()-_tt:.1f}s)')

            # Deduplicate boulders
            seen, dedup = {}, []
            for b in sorted(all_boulders, key=lambda x: -x['diameter_m']):
                key = (b['x_pixel']//15, b['y_pixel']//15)
                if key not in seen: seen[key]=True; dedup.append(b)
            all_boulders = dedup
            print(f'  ⏱ Detection: {time.time()-_t:.1f}s  |  B:{len(all_boulders)} L:{len(all_landslides)}')

            # ── Age classification ────────────────────────────────────────
            _t = time.time()
            classified_landslides = []
            for ls in all_landslides:
                t_img2  = ls.pop('_tile_img', img)
                t_dem2  = ls.pop('_tile_dem', dem)
                row_off2= ls.pop('_row_off',  0)
                col_off2= ls.pop('_col_off',  0)
                ls_local = ls.copy()
                ls_local['coords']     = ls['coords'] - np.array([[row_off2,col_off2]])
                ls_local['source_row'] = ls['source_row'] - row_off2
                ls_local['source_col'] = ls['source_col'] - col_off2
                classified = compute_freshness_score(t_img2, t_dem2, ls_local,
                                                      CFG['age_classification'], PIXEL_RES)
                classified['coords']       = ls['coords']
                classified['source_row']   = ls['source_row']
                classified['source_col']   = ls['source_col']
                classified['centroid_row'] = ls['centroid_row']
                classified['centroid_col'] = ls['centroid_col']
                classified_landslides.append(classified)
            all_boulders = tag_boulders_near_landslides(all_boulders, classified_landslides)
            n_recent  = sum(1 for l in classified_landslides if l.get('age_class')=='recent')
            n_ancient = len(classified_landslides) - n_recent
            print(f'  ⏱ Age classification: {time.time()-_t:.1f}s  |  Recent:{n_recent} Ancient:{n_ancient}')

            # ── Export CSVs ───────────────────────────────────────────────
            _t = time.time()
            b_rows = [{'x_pixel_global':b['x_pixel'],'y_pixel_global':b['y_pixel'],
                       'radius_px':b['radius_px'],'diameter_m':b['diameter_m'],
                       'contrast':b.get('contrast',0),'near_landslide':b.get('near_landslide',False),
                       'landslide_age':b.get('landslide_age','none')} for b in all_boulders]
            bdf = pd.DataFrame(b_rows).drop_duplicates(
                subset=['x_pixel_global','y_pixel_global']
            ).sort_values('diameter_m', ascending=False).reset_index(drop=True)
            bdf.to_csv(os.path.join(csv_dir, 'boulders.csv'), index=False)

            l_rows = [{'source_col_global':ls['source_col'],'source_row_global':ls['source_row'],
                       'area_m2':ls.get('area_m2',0),'major_axis_m':ls.get('major_axis_m',0),
                       'minor_axis_m':ls.get('minor_axis_m',0),'elongation':ls.get('elongation',0),
                       'freshness_score':ls.get('freshness_score',0),'age_class':ls.get('age_class','?')}
                      for ls in classified_landslides]
            ldf = pd.DataFrame(l_rows).sort_values('area_m2',ascending=False).reset_index(drop=True)
            ldf.to_csv(os.path.join(csv_dir, 'landslides.csv'), index=False)
            print(f'  ⏱ CSV export: {time.time()-_t:.2f}s')

            # ── Annotated map ─────────────────────────────────────────────
            _t = time.time()
            h_v, w_v = img.shape
            base_rgb  = _to_rgb(img)
            annotated = draw_landslide_fills(base_rgb, classified_landslides, vcfg)
            annotated = draw_boulders_overlay(annotated, all_boulders, vcfg)

            # Full annotated map
            fig, axes = plt.subplots(1, 2, figsize=(20, 10))
            axes[0].imshow(img, cmap='gray', interpolation='nearest')
            axes[0].set_title('Original', fontsize=13, fontweight='bold'); axes[0].axis('off')
            axes[1].imshow(annotated, interpolation='nearest')
            axes[1].set_title('Annotated Detections', fontsize=13, fontweight='bold'); axes[1].axis('off')
            handles = [
                mpatches.Patch(color=np.array(vcfg['boulder_color'])/255,  label=f'Boulders ({len(bdf)})'),
                mpatches.Patch(color=np.array(vcfg['recent_color'])/255,   label=f'Recent ({n_recent})'),
                mpatches.Patch(color=np.array(vcfg['ancient_color'])/255,  label=f'Ancient ({n_ancient})'),
                mpatches.Patch(color=np.array(vcfg['source_color'])/255,   label='Source Points'),
            ]
            fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=10, framealpha=0.85)
            fig.suptitle(f'{img_name}  |  Boulders: {len(bdf)}  Landslides: {len(ldf)} (R:{n_recent} A:{n_ancient})',
                         fontsize=12)
            plt.tight_layout(rect=[0,0.04,1,0.97])
            full_map_path = os.path.join(maps_dir, f'{img_name}_annotated.png')
            plt.savefig(full_map_path, dpi=vcfg['dpi'], bbox_inches='tight', facecolor='black')
            plt.show()
            plt.close()

            # Cropped map (tight bbox around detections)
            all_xs = [b['x_pixel'] for b in all_boulders] + [int(ls['source_col']) for ls in classified_landslides]
            all_ys = [b['y_pixel'] for b in all_boulders] + [int(ls['source_row']) for ls in classified_landslides]
            if all_xs and all_ys:
                pad = max(60, int((max(all_xs)-min(all_xs))*0.06))
                x1 = max(0, min(all_xs)-pad); y1 = max(0, min(all_ys)-pad)
                x2 = min(w_v, max(all_xs)+pad); y2 = min(h_v, max(all_ys)+pad)
                crop = annotated[y1:y2, x1:x2]
                fig2, ax2 = plt.subplots(figsize=(14,9))
                ax2.imshow(crop, interpolation='nearest')
                ax2.set_title(f'{img_name} — Cropped Detection Zone ({x2-x1}x{y2-y1}px)', fontsize=12)
                ax2.axis('off')
                plt.tight_layout()
                crop_path = os.path.join(maps_dir, f'{img_name}_cropped.png')
                plt.savefig(crop_path, dpi=vcfg['dpi'], bbox_inches='tight')
                plt.show()
                plt.close()

            # Save input preview
            fig3, axes3 = plt.subplots(1,2,figsize=(12,5))
            axes3[0].imshow(img, cmap='gray'); axes3[0].set_title('OHRC Image'); axes3[0].axis('off')
            axes3[1].imshow(dem, cmap='terrain'); axes3[1].set_title('Elevation'); axes3[1].axis('off')
            plt.suptitle(img_name); plt.tight_layout()
            plt.savefig(os.path.join(maps_dir, f'{img_name}_input_preview.png'), dpi=100, bbox_inches='tight')
            plt.close()

            t_total = time.time() - t_img_start
            print(f'  ⏱ Visualization: {time.time()-_t:.1f}s')
            print(f'  ✅ Done in {t_total:.1f}s')
            print(f'     outputs/{img_name}/csv/boulders.csv       ({len(bdf)} rows)')
            print(f'     outputs/{img_name}/csv/landslides.csv     ({len(ldf)} rows)')
            print(f'     outputs/{img_name}/maps/{img_name}_annotated.png')
            print(f'     outputs/{img_name}/maps/{img_name}_cropped.png')

            batch_summary.append({
                'image': img_name,
                'size':  f'{img.shape[1]}x{img.shape[0]}',
                'boulders': len(bdf),
                'landslides': len(ldf),
                'recent': n_recent,
                'ancient': n_ancient,
                'time_s': round(t_total, 1),
                'status': 'OK'
            })

        except Exception as e:
            import traceback
            print(f'  ❌ ERROR: {e}')
            traceback.print_exc()
            batch_summary.append({'image':img_name,'status':f'ERROR: {e}',
                                   'boulders':0,'landslides':0,'recent':0,'ancient':0,'time_s':0})

        print()

    total_time = time.time() - t_batch_start
    print(f'{'='*60}')
    print(f'  BATCH COMPLETE — {total_time:.1f}s total')
    print(f'{'='*60}')


  BATCH START — 11 image(s) to process

[1/11] ohrc_image_01
  Output → outputs/ohrc_image_01
    Resizing 1200x9015 → 136x1023 (scale=0.11)
  ⏱ Load: 1.1s  |  Size: 136x1023px
  ⏱ Tiling: 0.00s  |  2 tiles
| B:557 L:14 (1.4s)
    Tile 2/2 | B:29 L:0 (0.0s)
  ⏱ Detection: 1.5s  |  B:421 L:14
  ⏱ Age classification: 0.6s  |  Recent:0 Ancient:14
  ⏱ CSV export: 0.00s
  ⏱ Visualization: 2.1s
  ✅ Done in 5.3s
     outputs/ohrc_image_01/csv/boulders.csv       (421 rows)
     outputs/ohrc_image_01/csv/landslides.csv     (14 rows)
     outputs/ohrc_image_01/maps/ohrc_image_01_annotated.png
     outputs/ohrc_image_01/maps/ohrc_image_01_cropped.png

[2/11] ohrc_image_02
  Output → outputs/ohrc_image_02
    Resizing 1200x9369 → 131x1024 (scale=0.11)
  ⏱ Load: 1.2s  |  Size: 131x1024px
  ⏱ Tiling: 0.00s  |  2 tiles
| B:733 L:20 (1.8s)
    Tile 2/2 | B:46 L:1 (0.0s)
  ⏱ Detection: 1.9s  |  B:455 L:21
  ⏱ Age classification: 0.9s  |  Recent:0 Ancient:21
  ⏱ CSV export: 0.00s
  ⏱ Visualization: 2.3s

## 📊 Step 6 — Batch Summary

In [21]:
if batch_summary:
    summary_df = pd.DataFrame(batch_summary)
    print('\n📋 BATCH RESULTS SUMMARY')
    print('='*70)
    display(summary_df)

    # Save combined summary CSV
    summary_path = os.path.join(OUTPUT_FOLDER, 'batch_summary.csv')
    summary_df.to_csv(summary_path, index=False)
    print(f'\n✅ Summary saved → {summary_path}')

    # Print folder tree
    print('\n📁 Output folder structure:')
    for row in batch_summary:
        name = row['image']
        print(f'  outputs/{name}/')
        print(f'  ├── csv/')
        print(f'  │   ├── boulders.csv    ({row["boulders"]} rows)')
        print(f'  │   └── landslides.csv  ({row["landslides"]} rows)')
        print(f'  └── maps/')
        print(f'      ├── {name}_annotated.png')
        print(f'      ├── {name}_cropped.png')
        print(f'      └── {name}_input_preview.png')
        print()
else:
    print('No images were processed.')



📋 BATCH RESULTS SUMMARY


,image,size,boulders,landslides,recent,ancient,time_s,status
0,ohrc_image_01,136x1023,421,14,0,14,5.3,OK
1,ohrc_image_02,131x1024,455,21,0,21,6.3,OK
2,ohrc_image_03,153x1024,237,18,0,18,5.0,OK
3,ohrc_image_04,160x1024,283,30,1,29,6.0,OK
4,ohrc_image_05,136x1023,224,18,1,17,4.3,OK
5,ohrc_image_06,640x640,390,38,23,15,16.9,OK
6,ohrc_image_07.,640x640,776,27,15,12,22.5,OK
7,ohrc_image_08.,640x640,521,20,8,12,16.7,OK
8,ohrc_image_09,640x640,1054,31,9,22,28.2,OK
9,ohrc_image_10,640x640,1166,52,39,13,32.4,OK



✅ Summary saved → outputs/batch_summary.csv

📁 Output folder structure:
  outputs/ohrc_image_01/
  ├── csv/
  │   ├── boulders.csv    (421 rows)
  │   └── landslides.csv  (14 rows)
  └── maps/
      ├── ohrc_image_01_annotated.png
      ├── ohrc_image_01_cropped.png
      └── ohrc_image_01_input_preview.png

  outputs/ohrc_image_02/
  ├── csv/
  │   ├── boulders.csv    (455 rows)
  │   └── landslides.csv  (21 rows)
  └── maps/
      ├── ohrc_image_02_annotated.png
      ├── ohrc_image_02_cropped.png
      └── ohrc_image_02_input_preview.png

  outputs/ohrc_image_03/
  ├── csv/
  │   ├── boulders.csv    (237 rows)
  │   └── landslides.csv  (18 rows)
  └── maps/
      ├── ohrc_image_03_annotated.png
      ├── ohrc_image_03_cropped.png
      └── ohrc_image_03_input_preview.png

  outputs/ohrc_image_04/
  ├── csv/
  │   ├── boulders.csv    (283 rows)
  │   └── landslides.csv  (30 rows)
  └── maps/
      ├── ohrc_image_04_annotated.png
      ├── ohrc_image_04_cropped.png
      └── ohrc_ima

## ✅ All Done!

Each image in your input folder now has its own output subfolder:

```
outputs/
├── batch_summary.csv          ← Summary of all images
├── image_01/
│   ├── csv/
│   │   ├── boulders.csv
│   │   └── landslides.csv
│   └── maps/
│       ├── image_01_annotated.png
│       ├── image_01_cropped.png
│       └── image_01_input_preview.png
├── image_02/
│   └── ...
```
